# 酒店预订数据概览与清洗

**项目目标**：基于 11.9 万条酒店预订记录，使用 Pandas 完成数据读取、缺失值处理、衍生字段构造与异常值检查。

**本 Notebook 内容**：
1. 项目环境配置与 src 导入
2. 读取原始数据
3. 数据概览（shape / dtypes / describe / 唯一值）
4. 缺失值统计与可视化
5. 执行缺失值处理
6. 构造衍生字段
7. 异常值检查（ADR <= 0、无效入住人数等）
8. 保存清洗后数据
9. 小结与下一步

> **注意**：`set_plot_style()` 已修复中文字体配置，旧的乱码图片已删除。请 **重新运行第 3 步的 `plot_missing_values` + `save_figure` 单元格**，以重新生成 `01_missing_values.png`。

In [4]:
# 确保从 notebooks 目录运行时也能正确导入 src
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

print("当前工作目录：", Path.cwd())

当前工作目录： c:\Users\Lenovo\Desktop\酒店项目\pandas-hotel-booking-analysis


In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 导入项目 src 模块
from src.config import RAW_DATA_PATH, PROCESSED_DATA_PATH, FIGURE_DIR
from src.data_cleaning import load_data, get_data_overview, get_missing_summary, clean_missing_values
from src.feature_engineering import create_features
from src.visualization import set_plot_style, save_figure, plot_missing_values, plot_cancel_rate_by_group

# 设置绘图风格
set_plot_style()

print("导入完成。")

[字体配置] rcParams 已设置为: Microsoft YaHei
导入完成。


---
## 第 1 步：读取原始数据

In [6]:
df = load_data(RAW_DATA_PATH)
print(f"数据集大小: {df.shape[0]:,} 行 × {df.shape[1]} 列")
print(f"数据文件路径: {RAW_DATA_PATH}")

数据集大小: 119,390 行 × 32 列
数据文件路径: data/raw/hotel_bookings.csv


---
## 第 2 步：数据概览

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [8]:
df.head(5)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [9]:
# 数值列描述统计
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [10]:
# 调用 src 中的完整概览函数
get_data_overview(df)

【数据概览】
Shape: (119390, 32)
内存占用: 104.83 MB

列名与数据类型:
hotel                              object
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                 object
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                               object
country                            object
market_segment                     object
distribution_channel               object
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                 object
assigned_room_type                 object
booking_changes                     int64
deposit_type           

---
## 第 3 步：缺失值统计与可视化

In [11]:
# 获取缺失值统计表
missing_summary = get_missing_summary(df)
print("缺失值统计：")
missing_summary

缺失值统计：


,列名,缺失数量,缺失比例(%)
24,company,112593,94.31
23,agent,16340,13.69
13,country,488,0.41
10,children,4,0.00


In [12]:
# 绘制缺失值柱状图
fig_missing = plot_missing_values(missing_summary)
save_figure(fig_missing, "01_missing_values.png")

图表已保存: outputs/figures\01_missing_values.png


**缺失值处理策略**：
- `company`：缺失率 94.3%，几乎全空 → 创建 `has_company` 布尔标识后删除
- `agent`：缺失率 13.7%，缺失代表无中介 → 填充 0，创建 `has_agent` 布尔标识
- `country`：缺失率 0.41% → 填充 "Unknown"
- `children`：4 个缺失 → 填充 0，转为 int

---
## 第 4 步：执行缺失值处理

In [13]:
df_clean = clean_missing_values(df)

# 验证：确认清洗后无缺失值
missing_after = df_clean.isnull().sum().sum()
print(f"清洗后缺失值总数: {missing_after}")
print(f"清洗后数据大小: {df_clean.shape[0]:,} 行 × {df_clean.shape[1]} 列")

清洗后缺失值总数: 0
清洗后数据大小: 119,390 行 × 33 列


In [14]:
# 确认清洗前后对比
print("清洗前列:", df.shape[1], "→ 清洗后列:", df_clean.shape[1])
print("新增列: has_company, has_agent")
print("删除列: company")
print(f"agent 缺失已填充为 0，原缺失 {missing_summary.loc[missing_summary['列名']=='agent', '缺失数量'].values[0]:,} 条")

清洗前列: 32 → 清洗后列: 33
新增列: has_company, has_agent
删除列: company
agent 缺失已填充为 0，原缺失 16,340 条


---
## 第 5 步：构造衍生字段

In [15]:
df_feat = create_features(df_clean)

# 列出新增字段
new_cols = ["arrival_date", "total_nights", "total_guests", "is_family",
            "lead_time_group", "adr_level", "season", "room_match", "is_valid_guest"]
print("新增衍生字段:")
for col in new_cols:
    print(f"  - {col} ({df_feat[col].dtype})")

新增衍生字段:
  - arrival_date (datetime64[ns])
  - total_nights (int64)
  - total_guests (int64)
  - is_family (int32)
  - lead_time_group (category)
  - adr_level (object)
  - season (object)
  - room_match (int32)
  - is_valid_guest (int32)


In [16]:
# 查看 lead_time_group 分布（确认 lead_time=0 被正确归类）
print("lead_time_group 分布:")
print(df_feat["lead_time_group"].value_counts().sort_index())
print()
print(f"lead_time=0 的记录数: {(df_feat['lead_time'] == 0).sum()}")
print(f"lead_time=0 且被归入 '0-7天' 的记录数: {((df_feat['lead_time'] == 0) & (df_feat['lead_time_group'] == '0-7天')).sum()}")

lead_time_group 分布:
0-7天       19746
8-30天      18960
31-90天     29553
91-180天    26439
180天以上     24692
Name: lead_time_group, dtype: int64

lead_time=0 的记录数: 6345
lead_time=0 且被归入 '0-7天' 的记录数: 6345


In [17]:
# 查看 adr_level 分布（确认 ADR <= 0 被标记为异常）
print("adr_level 分布:")
print(df_feat["adr_level"].value_counts())
print()
print(f"ADR <= 0 的记录数: {(df_feat['adr'] <= 0).sum()}")

adr_level 分布:
低        39283
高        39128
中        39019
异常/免费     1960
Name: adr_level, dtype: int64

ADR <= 0 的记录数: 1960


---
## 第 6 步：异常值检查

In [18]:
# 检查 ADR 异常值（<= 0 或极端值）
adr_zero = (df_feat["adr"] <= 0).sum()
adr_extreme_high = df_feat[df_feat["adr"] > 0]["adr"].describe()
print(f"ADR <= 0 的记录数: {adr_zero} ({adr_zero/len(df_feat)*100:.2f}%)")
print(f"ADR > 0 的统计描述:")
print(adr_extreme_high)

ADR <= 0 的记录数: 1960 (1.64%)
ADR > 0 的统计描述:
count    117430.000000
mean        103.530818
std          49.198721
min           0.260000
25%          70.530000
50%          95.000000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64


In [19]:
# 检查入住人数为 0 的记录
guests_zero = (df_feat["total_guests"] == 0).sum()
print(f"total_guests == 0 的记录数: {guests_zero}")
if guests_zero > 0:
    print("示例:")
    print(df_feat[df_feat["total_guests"] == 0].head(3).to_string())

total_guests == 0 的记录数: 180
示例:
             hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  arrival_date_week_number  arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  adults  children  babies meal country market_segment distribution_channel  is_repeated_guest  previous_cancellations  previous_bookings_not_canceled reserved_room_type assigned_room_type  booking_changes deposit_type  agent  days_in_waiting_list    customer_type  adr  required_car_parking_spaces  total_of_special_requests reservation_status reservation_status_date  has_company  has_agent arrival_date  total_nights  total_guests  is_family lead_time_group adr_level season  room_match  is_valid_guest
2224  Resort Hotel            0          1               2015            October                        41                          6                        0                     3       0         0       0   SC     PRT      Corporate            Corporate                  0        

In [20]:
# 检查 lead_time 极端值
print("lead_time 分布:")
print(df_feat["lead_time"].describe())
print()
lead_extreme = df_feat[df_feat["lead_time"] > 700]
print(f"lead_time > 700 天的记录数: {len(lead_extreme)} ({len(lead_extreme)/len(df_feat)*100:.2f}%)")

lead_time 分布:
count    119390.000000
mean        104.011416
std         106.863097
min           0.000000
25%          18.000000
50%          69.000000
75%         160.000000
max         737.000000
Name: lead_time, dtype: float64

lead_time > 700 天的记录数: 2 (0.00%)


In [21]:
# 综合异常摘要
is_clean = (
    (df_feat["total_guests"] > 0) & (df_feat["adr"] > 0) & (df_feat["is_valid_guest"] == 1)
)
print(f"有效记录（total_guests > 0 且 adr > 0）: {is_clean.sum():,} 条")
print(f"无效记录: {(~is_clean).sum():,} 条")

有效记录（total_guests > 0 且 adr > 0）: 117,399 条
无效记录: 1,991 条


---
## 第 7 步：保存清洗后数据

In [22]:
# 确保输出目录存在
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

# 保存清洗并增强后的数据
df_feat.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"清洗后数据已保存到: {PROCESSED_DATA_PATH}")
print(f"数据大小: {df_feat.shape[0]:,} 行 × {df_feat.shape[1]} 列")
print(f"文件大小: {os.path.getsize(PROCESSED_DATA_PATH) / 1024**2:.2f} MB")

清洗后数据已保存到: data/processed/hotel_bookings_cleaned.csv
数据大小: 119,390 行 × 42 列
文件大小: 20.93 MB


---
## 小结

本 Notebook 完成了以下工作：

1. **数据读取**：成功加载 119,390 条 × 32 列的原始预订数据
2. **缺失值处理**：
   - `company` 缺失率 94.3% → 转为 `has_company` 布尔标识后删除
   - `agent` 缺失率 13.7% → 填充 0 + 新建 `has_agent`
   - `country` 缺失率 0.41% → 填充 "Unknown"
   - `children` 4 条缺失 → 填充 0
   - `reservation_status_date` → 转为 datetime
3. **衍生字段**：构造了 9 个分析用字段（`arrival_date`、`total_nights`、`total_guests`、`is_family`、`lead_time_group`、`adr_level`、`season`、`room_match`、`is_valid_guest`）
4. **异常值检查**：识别了 ADR <= 0 的记录及无效入住记录
5. **数据保存**：清洗后数据已保存至 `data/processed/hotel_bookings_cleaned.csv`

**下一步**：使用清洗后的数据进行取消行为分析（notebooks/02_cancellation_analysis.ipynb）。